In [1]:
import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.optim as optim

import rlenvs
# from scipy.special import softmax
# import pandas as pd


In [2]:
default_params = {
    'gravity': 9.8,
    'masscart': 1.0,
    'masspole': 0.1,
    'length': 0.5,  # actually half the pole's length
    'force_mag': 10.0,
    'tau': 0.02,  # seconds between state updates
    # 'total_mass': self.masspole + self.masscart,
    # 'polemass_length': self.masspole * self.length,
}

# Geração de Dados

In [3]:
def generate_hist(n=500):
    # env = gym.make("CartPole-v1")
    # observation, info = env.reset(seed=82)
    env = gym.make("custom/DiscreteCartPole-v1")
    options = {
        'masspole': round(np.random.rand(), 2),
        'length': np.random.randint(low=0, high=20)/10
    }
    observation, info = env.reset(seed=82, options=options)

    hist_s = np.array([observation])
    hist_a = np.array([])
    hist_p = np.array([list(options.values())])
    for _ in range(n):
        action = int(np.random.choice([0,1], size=1)[0])
        observation,reward, terminated, truncated, info = env.step(action)
        
        hist_s = np.concat([hist_s, [observation]])
        hist_a = np.concat([hist_a, [action]])
        hist_p = np.concat([hist_p, [list(options.values())]])

        if terminated or truncated:
            # break
            options = {
                'masspole': round(np.random.rand(), 2),
                'length': np.random.randint(low=0, high=20)/10
            }
            observation, info = env.reset(options=options)
            
    env.close()
    return hist_s, hist_a, hist_p

# [(s,a) for s,a in zip(hist_s, hist_a)]
# hist_a.sum()
hist_s, hist_a, hist_p = generate_hist()
hist_s

/home/gui/Área de Trabalho/rlalgs/.venv/lib/python3.9/site-packages/gymnasium/utils/passive_env_checker.py:168: DeprecationWarning: WARN: Current gymnasium version requires that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.deprecation(
/home/gui/Área de Trabalho/rlalgs/.venv/lib/python3.9/site-packages/gymnasium/utils/passive_env_checker.py:181: DeprecationWarning: WARN: Current gymnasium version requires that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.deprecation(
/home/gui/Área de Trabalho/rlalgs/.venv/lib/python3.9/site-packages/rlenvs/cart_pole_custom.py:126: RuntimeWarning: divide by zero encountered in scalar divide
  thetaacc = (self.gravity * sintheta - costheta * temp) / (
/home/gui/Área de Trabalho/rlalgs/.venv/lib/python3.9/site-packages/rlenvs/cart_pole_custom.py:130: RuntimeWarning: invalid value encountered in sca

array([[ 0.     ,  0.     , -0.02549,  0.05352],
       [ 0.     ,  0.     , -0.02129,  0.20287],
       [ 0.     ,  0.     , -0.01714,  0.3735 ],
       ...,
       [ 0.     ,  0.     , -0.08245, -0.30882],
       [ 0.     ,  0.     , -0.08552, -0.20029],
       [ 0.     ,  0.     , -0.08552, -0.09117]], dtype=float32)

# Estimador de Estados

In [12]:
input_a.shape

(500, 1)

In [13]:
p = .3

input_s = hist_s[:-2,2:] # Getting ony two dimensions of state
input_s_ = hist_s[1:-1,2:] # Getting ony two dimensions of state
input_s_2 = hist_s[2:,2:] # Getting ony two dimensions of state
input_a = np.expand_dims(hist_a[:-1], axis=1)
input_a_ = np.expand_dims(hist_a[1:], axis=1)
input_p = hist_p[:-2]
input_all = np.concat([input_s, input_a, input_s_, input_a_], axis=1)

split = int(input_all.shape[0] * p)

X = torch.tensor(input_all, dtype=torch.float32)
y = torch.tensor(input_s_2, dtype=torch.float32)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

In [23]:
class NeuralNet(nn.Module):
    def __init__(self, s,a,p, hidden_size, output):
        super(NeuralNet, self).__init__()
        self.par1 = nn.Linear(sum([s,a,s,a]), hidden_size)
        self.par2 = nn.Linear(hidden_size, p)
        self.s1 = nn.Linear(sum([s,a,p]), hidden_size)
        self.s2 = nn.Linear(hidden_size, output)
        self.relu = nn.ReLU()
        self.than = nn.Tanh()

    def forward_param(self, x):
        out = self.par1(x)
        out = self.relu(out)
        out = self.par2(out)
        return out
    
    def forward_state(self, x):
        out = self.s1(x)
        out = self.than(out)
        out = self.s2(out)
        return out

    def forward(self, x):
        out = self.forward_param(x)
        sInput = torch.concat([x[:,:3], out], axis=1)
        out = self.forward_state(sInput)
        return out

model = NeuralNet(2,1,2, hidden_size=10, output=2)

In [25]:
class Model():
    def __init__(self,
                input_size = (2, 1, 2), # 2d for s and s_, 1d for a and a_, and 2d for p,
                hidden_size = 10,
                output = 2, #2d for s_
                learning_rate = 0.001,
                criterion_clss = nn.MSELoss,
                optimizer_clss = optim.Adam
            ) -> None:
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output = output
        self.learning_rate = learning_rate

        self.estimator = NeuralNet(*input_size, hidden_size, output)
        
        self.criterion = criterion_clss()
        self.optimizer = optimizer_clss(self.estimator.parameters(), lr=learning_rate)

    def train(self, X_train, y_train, num_epochs=100, debug=False):
        for epoch in range(num_epochs):
            outputs = self.estimator(X_train)
            loss = self.criterion(outputs, y_train)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()  

            if debug and (epoch+1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')  

    def predict(self, x):
        value = None
        with torch.no_grad():
            value = self.estimator(x)
        return value
    

m = Model()
m.train(X_train, y_train, num_epochs=5000, debug=False)

In [26]:
print('RMSE: ', torch.mean((m.predict(X_test) - y_test)**2, axis=0)**.5)
# print(m.predict(X_test))

RMSE:  tensor([0.0317, 0.0907])
